<a href="https://colab.research.google.com/github/JDaviA/Desafio/blob/main/Reconhecimento_Facial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import cv2
import numpy as np
import tensorflow as tf
import os
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

DATASET_DIR = "faces_dataset"
MODEL_PATH = "face_recognition_model.h5"
if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)

cap = cv2.VideoCapture(0)
labels = []
faces_data = []

print("Iniciando captura de imagens e reconhecimento facial. Pressione 'q' para sair.")

if os.path.exists(MODEL_PATH):
    model = load_model(MODEL_PATH)
    print("Modelo carregado para reconhecimento facial.")
else:
    model = None
    print("Nenhum modelo encontrado. Coletando dados para treinamento.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4)

    for (x, y, w, h) in faces:
        face = gray[y:y+h, x:x+w]
        face = cv2.resize(face, (100, 100))
        face_array = np.array(face).reshape(-1, 100, 100, 1) / 255.0

        if model:
            prediction = model.predict(face_array)
            label = np.argmax(prediction)
            confidence = np.max(prediction)
            label_text = f"Pessoa {label} ({confidence:.2f})"
        else:
            label_text = "Coletando dados..."
            faces_data.append(face)
            labels.append(0)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
        cv2.putText(frame, label_text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.imshow('Reconhecimento Facial', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Sem modelo, treinar com dados coletados
if not model and len(faces_data) > 0:
    faces_data = np.array(faces_data).reshape(-1, 100, 100, 1) / 255.0
    labels = to_categorical(np.array(labels), num_classes=2)

    split_size = 0.1 if len(faces_data) > 10 else 0.0
    X_train, X_test, y_train, y_test = train_test_split(faces_data, labels, test_size=split_size, random_state=42)

    # Criar modelo CNN
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(100, 100, 1)),
        MaxPooling2D((2,2)),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D((2,2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(2, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    if len(X_train) > 0:
        model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))
        model.save(MODEL_PATH)
        print("Treinamento concluído e modelo salvo!")
    else:
        print("Dados insuficientes para treinamento. Colete mais imagens e tente novamente.")


Iniciando captura de imagens e reconhecimento facial. Pressione 'q' para sair.
Nenhum modelo encontrado. Coletando dados para treinamento.
